# 05 — String-Level Disagreement Analysis

This notebook characterises how much services agree on translations and, when they disagree, what *kind* of disagreement appears in the output strings. The goal is not to impose a definitive taxonomy of translation behavior, but to surface recurring disagreement patterns using only string-level evidence and a small amount of external metadata (Wikipedia presence, source-term borrowing detectable in the string).

The result is therefore best read as a **rule-based triage layer**: a transparent way to sort languages into analytically useful buckets before closer inspection, not a claim that each label captures a single underlying linguistic reality.

**Methodological note — no rationale keywords.**  
The classifier deliberately excludes the models' rationale text from the canonical classification signal. This keeps the disagreement labels independent of the rationale analysis in notebook 07, avoiding circularity. The models' own reasoning is analysed separately as an independent axis in that notebook.

Five categories, ordered from strongest consensus signal to weakest external anchoring:

| Category | String signal |
|---|---|
| `COMPLETE_CONSENSUS` | All services produced the same string — no disagreement |
| `MEASUREMENT_ARTEFACT` | Disagreement collapses after normalisation (case / diacritics / script) |
| `PRODUCTIVE_DISAGREEMENT` | Multiple distinct outputs AND Wikipedia translation exists |
| `STRUCTURAL_ABSENCE` | At least one service borrowed the source term unadapted |
| `TRANSMOGRIFICATION` | Multiple distinct outputs, no Wikipedia anchor, no borrowing |

The last two labels are especially heuristic. `STRUCTURAL_ABSENCE` often means "some services fall back to source-term borrowing," not that absence has been conclusively demonstrated. `TRANSMOGRIFICATION` is the residual class for unanchored divergence: it captures outputs that remain difficult to interpret with the available evidence, ranging from plausible local coinages to clearly unstable model invention.

Input: `translated_terms/digital_humanities/evaluation/across_variant_detail.csv`  
Output: `translated_terms/digital_humanities/evaluation/disagreement_analysis.csv`

**Exclusion tier: Tier 2 — Translation Analysis** (see [docs/exclusion_strategy.md](../docs/exclusion_strategy.md))

Term-error flags (`has_mixed_script`, `has_placeholder_term`, `has_repetition_loop`, `has_extreme_term_length`, `has_unicode_escape`) are used to drop individual service terms before classification. Manual `exclude_translation=True` entries are also applied. `has_source_term` and `has_script_disagreement` are retained — borrowing and cross-service script divergence are classification inputs, not errors.

Just as important, the notebook compares the data in **stages** rather than treating the raw strings as immediately commensurable. It first removes unusable service-level outputs, then builds both a raw-string view and a normalized-string view, and only then asks whether remaining differences are superficial, anchored, borrowing-based, or still unaccounted for.



## 5.1 Setup, Cleaning, and Run

This section runs the canonical string-level classifier and writes `disagreement_analysis.csv`. The canonical pass keeps the rule chain grounded in output strings and external anchoring signals, while leaving rationale-keyword rules for the sensitivity check in §5.4.

The comparison pipeline works by **progressive normalization** rather than by a single binary clean/dirty decision:

1. **Load a single comparison layer**: the notebook starts from `across_variant_detail.csv` plus one chosen rationale variant (`minimal` here), so every language is compared on the same service grid.
2. **Enforce translation-rationale pairing**: upstream loading nulls out translations with missing or placeholder rationales, and vice versa. Within the disagreement script, any remaining LLM translation that lacks a usable rationale in the chosen variant is dropped before comparison.
3. **Remove obvious term-level failures**: automated review signals and manual exclusions suppress service outputs that are not analytically meaningful for string comparison, such as mixed-script corruption, placeholders, repetition loops, extreme term length, or unicode-escape artefacts.
4. **Retain analytic signals rather than excluding them**: source-term borrowing and script disagreement are *not* treated as errors here, because they are part of what the classifier is trying to detect.
5. **Construct two views of each language's outputs**: a raw-string view (`n_unique_raw`) and a normalized-string view (`n_unique_normalized`). Normalization lowercases, strips punctuation and diacritics, and collapses whitespace so that superficial surface variation does not automatically count as substantive disagreement.
6. **Measure residual distance after normalization**: the classifier computes pairwise edit distance among normalized forms and uses a length-sensitive threshold to decide whether the remaining variation is still small enough to count as artefactual.
7. **Only then apply the heuristic buckets**: after cleaning, normalization, and distance checks, the script asks whether the remaining disagreement is externally anchored (Wikipedia), visibly borrowing the source term, or left as unanchored divergence.

That sequence matters because the notebook is not comparing raw model strings all at once. It is gradually reducing noise so that later category labels rest on a narrower and more interpretable residue of disagreement.



In [ ]:
import ast
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.enable('vegafusion')

sys.path.insert(0, str(Path('..').resolve()))
from scripts.utils import get_data_directory_path, read_csv_file
from scripts.exploration.explore_disagreements import (
    ClassifierConfig,
    CATEGORIES,
    derive_source_tokens,
    find_loan_words,
    run_disagreement_analysis,
    load_exclusions,
)



In [ ]:
DATA_DIR         = get_data_directory_path()
TERM_SLUG        = 'digital_humanities'
EVAL_DIR         = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')
DISAGR_CSV       = os.path.join(EVAL_DIR, 'disagreement_analysis.csv')
DISAGR_NO_KW_CSV = os.path.join(EVAL_DIR, 'disagreement_analysis_no_keywords.csv')
DISAGR_KW_CSV    = os.path.join(EVAL_DIR, 'disagreement_analysis_keywords.csv')

print(f'Eval dir       : {EVAL_DIR}')
print(f'Canonical CSV  : {DISAGR_CSV}')
print(f'No-keywords CSV: {DISAGR_NO_KW_CSV}')
print(f'Keywords CSV   : {DISAGR_KW_CSV}')
EXCL_PATH        = os.path.join(EVAL_DIR, 'manual_exclusions.csv')
FLAGS_PATH       = os.path.join(EVAL_DIR, 'automated_review_signals.csv')


In [ ]:
# ── Tier 2 exclusions: automated review signals + manual exclusions ────────────────────
# Term-error flags are converted to the same (language × service) exclusion
# format that run_disagreement_analysis already understands.

quality_flags_df = read_csv_file(FLAGS_PATH)

# ── Language count sanity check (pre-exclusion) ─────────────────────────────
_n = quality_flags_df['language_code'].nunique()
print(f'Languages loaded (pre-exclusion): {_n} (expected 880)')
if _n != 880:
    print(f'  ⚠ Expected 880 — re-run the translation pipeline or check for missing/duplicate language_code rows.')
else:
    print(f'  ✓ Count matches expected 880.')

TERM_ERROR_COLS = {
    'has_mixed_script':        'mixed_script_services',
    'has_placeholder_term':    'placeholder_term_services',
    'has_repetition_loop':     'repetition_loop_services',
    'has_extreme_term_length': 'extreme_term_length_services',
    'has_unicode_escape':      'unicode_escape_services',
}

def empty_entry():
    return {'term': False, 'rationale_minimal': False,
            'rationale_github_searcher': False,
            'rationale_fluent_speaker': False, 'rationale_judge': False}

tier2_excl = {}
for flag_col, svc_col in TERM_ERROR_COLS.items():
    for _, row in quality_flags_df[quality_flags_df[flag_col] == True].iterrows():
        lc = row['language_code']
        svcs = [s.strip() for s in str(row.get(svc_col, '')).split(';')
                if s.strip() and s.strip() != 'nan']
        for svc in svcs:
            tier2_excl.setdefault(lc, {}).setdefault(svc, empty_entry())['term'] = True

if os.path.exists(EXCL_PATH):
    manual_excl = load_exclusions(EXCL_PATH)
    for lc, svc_dict in manual_excl.items():
        for svc, entry in svc_dict.items():
            existing = tier2_excl.setdefault(lc, {}).setdefault(svc, empty_entry())
            for k, v in entry.items():
                existing[k] = existing.get(k, False) or v

n_term = sum(1 for svcs in tier2_excl.values() for e in svcs.values() if e['term'])
print(f'Tier 2: {n_term} (language × service) term exclusions')


In [ ]:
# ── Exclusion summary (uses new load_manual_exclusions utility) ──────────────
from scripts.utils import load_manual_exclusions as lme

analysis_langs, search_terms, _corrections = lme(EVAL_DIR)
print(f"Manual exclusions loaded:")
print(f"  analysis_exclusion : {len(analysis_langs)} language codes  (dropped from analysis)")
print(f"  search_exclusion   : {len(search_terms)} (language, term) pairs")
print(f"  term_correction    : {len(_corrections)} corrections")

In [ ]:
# Canonical run: string features + Wikipedia only, no rationale keyword rules.
# Writes disagreement_analysis_no_keywords.csv, then promoted to the canonical path
# read by notebooks 06 and 07.
cfg = ClassifierConfig(use_keyword_rules=False)

result_df = run_disagreement_analysis(
    data_directory_path=DATA_DIR,
    target_terms=['Digital Humanities'],
    rationale_variant='minimal',
    config=cfg,
    exclusions=tier2_excl,
)

result_df.to_csv(DISAGR_CSV, index=False)
print(f'\nCanonical CSV saved → {DISAGR_CSV}')
print(f'Rows: {len(result_df):,}  |  Columns: {len(result_df.columns)}')

## 5.2 Heuristic Disagreement Profile

How many languages fall into each rule-based bucket? This first pass is useful descriptively, but it should not be read as a fully resolved typology. In practice, the dominant split is between languages where at least one service falls back to visible source-term borrowing and languages where outputs diverge without an external anchor.



In [ ]:
CAT_ORDER = [
    'COMPLETE_CONSENSUS',
    'MEASUREMENT_ARTEFACT',
    'PRODUCTIVE_DISAGREEMENT',
    'STRUCTURAL_ABSENCE',
    'TRANSMOGRIFICATION',
]
CAT_COLORS = {
    'COMPLETE_CONSENSUS':      '#6baed6',
    'MEASUREMENT_ARTEFACT':    '#74c476',
    'PRODUCTIVE_DISAGREEMENT': '#fd8d3c',
    'STRUCTURAL_ABSENCE':      '#9e9ac8',
    'TRANSMOGRIFICATION':      '#de2d26',
}
CAT_LABELS = {
    'COMPLETE_CONSENSUS':      'Complete consensus (all services identical)',
    'MEASUREMENT_ARTEFACT':    'Surface variation (case / diacritics / script)',
    'PRODUCTIVE_DISAGREEMENT': 'Anchored disagreement (Wikipedia present)',
    'STRUCTURAL_ABSENCE':      'Borrowing / absence signal (source-term fallback)',
    'TRANSMOGRIFICATION':      'Unanchored divergence (residual bucket)',
}



In [ ]:
cat_df = result_df['category'].value_counts().reset_index()
cat_df['label'] = cat_df['category'].map(CAT_LABELS)
cat_df['pct']   = (cat_df['count'] / len(result_df) * 100).round(1)

dominant_n = int(
    cat_df.loc[cat_df['category'].isin(['STRUCTURAL_ABSENCE', 'TRANSMOGRIFICATION']), 'count'].sum()
)
dominant_pct = dominant_n / len(result_df) * 100

print('Category distribution:')
for _, r in cat_df.sort_values('count', ascending=False).iterrows():
    bar = '█' * int(r['count'] / len(result_df) * 40)
    print(f"  {r['category']:<30} {bar} {int(r['count']):>4}  ({r['pct']:.1f}%)")

print()
print(
    f"The two broad heuristic bins — STRUCTURAL_ABSENCE and TRANSMOGRIFICATION — account for "
    f"{dominant_n} / {len(result_df)} languages ({dominant_pct:.1f}%)."
)
print('That dominance is substantive: most of the notebook is separating visible borrowing from unanchored divergence, not resolving a fine-grained taxonomy.')

alt.Chart(cat_df).mark_bar().encode(
    x=alt.X('count:Q', title='Number of languages'),
    y=alt.Y('category:N', sort=CAT_ORDER, title=None),
    color=alt.Color(
        'category:N',
        scale=alt.Scale(domain=list(CAT_COLORS.keys()), range=list(CAT_COLORS.values())),
        legend=None,
    ),
    tooltip=[
        alt.Tooltip('label:N', title='Category'),
        alt.Tooltip('count:Q', title='Languages'),
        alt.Tooltip('pct:Q', format='.1f', title='%'),
    ],
).properties(
    width=520, height=220,
    title=alt.TitleParams(
        f'Heuristic disagreement profile — {len(result_df)} languages',
        subtitle='Rule-ordered buckets from strongest consensus signal to weakest external anchoring',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)



## 5.3 Rule Audit

The `rule_fired` column records which rule in the chain produced each verdict. This matters because the notebook is not inferring latent translation types from scratch; it is applying an ordered heuristic. With keyword rules disabled, `rule_sparse_coverage_absence` and `rule_convergent_absence` should show zero counts, confirming that the canonical run is resting on string features and external anchoring rather than rationale language.



In [ ]:
KEYWORD_RULES = {'rule_sparse_coverage_absence', 'rule_convergent_absence'}

rule_counts = result_df['rule_fired'].value_counts().reset_index()
rule_counts.columns = ['rule', 'n']
rule_counts['pct'] = (rule_counts['n'] / len(result_df) * 100).round(1)
rule_counts['is_keyword'] = rule_counts['rule'].isin(KEYWORD_RULES)

print('Rules that fired (keyword rules disabled — must be 0):')
for _, r in rule_counts.iterrows():
    flag = '  ← KEYWORD RULE (should be 0)' if r['is_keyword'] else ''
    print(f"  {r['rule']:<42} {r['n']:>4}  ({r['pct']:.1f}%){flag}")

alt.Chart(rule_counts).mark_bar().encode(
    x=alt.X('n:Q', title='Languages classified by this rule'),
    y=alt.Y('rule:N', sort='-x', title=None),
    color=alt.condition(
        alt.datum.is_keyword,
        alt.value('#de2d26'),
        alt.value('#4292c6'),
    ),
    tooltip=['rule:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='%')],
).properties(
    width=500, height=240,
    title=alt.TitleParams(
        'Rule-fired audit',
        subtitle='Red = keyword-dependent rules (should be 0 with use_keyword_rules=False)',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)

## 5.4 Keyword Rule Ablation

The classifier has two keyword-dependent rules — `rule_sparse_coverage_absence` and `rule_convergent_absence` — that fire when a rationale contains absence-signalling phrases ("untranslatable", "no equivalent", etc.). The canonical run above deliberately disables them to keep the classification grounded in string features alone, ensuring the disagreement labels remain independent of the rationale analysis in notebook 07.

This section is best read as a **sensitivity check**, not as a pass/fail test for the classifier. The question is whether rationale keywords materially re-sort the languages, and if so, which parts of the heuristic partition are most affected.

A small shift would suggest that the string-based canonical already captures most of the same signal. A larger shift would show that rationale language is doing nontrivial classificatory work and should remain analytically visible even if it is excluded from the canonical labels.



In [ ]:
cfg_kw = ClassifierConfig(use_keyword_rules=True)
kw_result_df = run_disagreement_analysis(
    data_directory_path=DATA_DIR,
    target_terms=['Digital Humanities'],
    rationale_variant='minimal',
    exclusions=tier2_excl,
    config=cfg_kw,
)
print(f'\nKeyword-enabled CSV: {DISAGR_KW_CSV}')

In [ ]:
no_kw_slim = pd.read_csv(DISAGR_NO_KW_CSV)[
    ['language_code', 'language_name', 'language_family', 'category', 'rule_fired']
].rename(columns={'category': 'cat_no_kw', 'rule_fired': 'rule_no_kw'})

kw_slim = pd.read_csv(DISAGR_KW_CSV)[
    ['language_code', 'category', 'rule_fired']
].rename(columns={'category': 'cat_kw', 'rule_fired': 'rule_kw'})

ablation_df = no_kw_slim.merge(kw_slim, on='language_code')
ablation_df['changed'] = ablation_df['cat_no_kw'] != ablation_df['cat_kw']

print(f'Total languages: {len(ablation_df)}')
print(f'Changed category when keywords enabled: {ablation_df["changed"].sum()} ({ablation_df["changed"].mean()*100:.1f}%)')
print()
print('Flow (no-kw → with-kw) for languages that change:')
flow = (
    ablation_df[ablation_df['changed']]
    .groupby(['cat_no_kw', 'cat_kw'])
    .size()
    .reset_index(name='n')
)
print(flow.to_string(index=False) if len(flow) else '  (none)')

In [ ]:
no_kw_counts = (
    ablation_df.groupby('cat_no_kw').size()
    .reset_index(name='n')
    .rename(columns={'cat_no_kw': 'category'})
    .assign(run='No keywords (canonical)')
)
kw_counts = (
    ablation_df.groupby('cat_kw').size()
    .reset_index(name='n')
    .rename(columns={'cat_kw': 'category'})
    .assign(run='With keywords')
)
compare_df = pd.concat([no_kw_counts, kw_counts])
compare_df['pct'] = (
    compare_df.groupby('run')['n']
    .transform(lambda x: x / x.sum() * 100)
    .round(1)
)

RUN_ORDER = ['No keywords (canonical)', 'With keywords']

alt.Chart(compare_df).mark_bar().encode(
    x=alt.X('category:N', sort=CAT_ORDER, title=None, axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('n:Q', title='Languages'),
    color=alt.Color(
        'run:N',
        sort=RUN_ORDER,
        scale=alt.Scale(domain=RUN_ORDER, range=['#4292c6', '#de2d26']),
        title='Run',
    ),
    xOffset=alt.XOffset('run:N', sort=RUN_ORDER),
    tooltip=['category:N', 'run:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='%')],
).properties(
    width=540, height=300,
    title='Category distribution: no-keywords (canonical) vs. keyword-enabled',
)

In [ ]:
flipped = (
    ablation_df[ablation_df['changed']]
    [['language_code', 'language_name', 'language_family', 'cat_no_kw', 'rule_no_kw', 'cat_kw', 'rule_kw']]
    .rename(columns={
        'cat_no_kw':  'category (no kw)',
        'rule_no_kw': 'rule (no kw)',
        'cat_kw':     'category (with kw)',
        'rule_kw':    'rule (with kw)',
    })
    .sort_values('category (with kw)')
)

n_flipped   = len(flipped)
pct_flipped = n_flipped / len(ablation_df) * 100

print(f'Languages that change category: {n_flipped} / {len(ablation_df)} ({pct_flipped:.1f}%)')
print()
if n_flipped == 0:
    print('Sensitivity read: enabling keyword rules does not re-sort any shared cases in this run.')
elif pct_flipped < 5:
    print(f'Sensitivity read: keyword rules make only a limited difference ({pct_flipped:.1f}% of shared cases change category).')
else:
    print(f'Sensitivity read: keyword rules materially re-sort the data ({pct_flipped:.1f}% of shared cases change category).')
print('Interpretation should focus on where the flips occur, not only on the headline percentage.')

if n_flipped > 0:
    print()
    print(flipped.to_string(index=False))



## 5.5 String Features per Category

These are the raw counts that drove the classification, including unique output counts and Levenshtein edit distances between normalised strings. They show what the classifier actually read, but they should still be interpreted heuristically: the labels compress a more continuous space of borrowing, convergence, and divergence.

Two distinctions are especially important here:

- **`n_unique_raw` vs `n_unique_normalized`**: the first counts distinct strings exactly as they appear after exclusions; the second counts distinct strings after lowercasing, punctuation removal, diacritic stripping, and whitespace collapse. The gap between the two is one way to see how much disagreement is merely surface-level.
- **Edit distance after normalization**: even when strings do not collapse to a single normalized form, the classifier still checks whether they remain only a few edits apart. That is what allows `MEASUREMENT_ARTEFACT` to absorb small orthographic differences rather than forcing them into a stronger disagreement bucket.

So this section is effectively the bridge between the cleaning pipeline in §5.1 and the category assignments that follow: it shows the reduced comparison space the rules are actually working on.



In [ ]:
feat_df = result_df[['category', 'n_unique_raw', 'n_unique_normalized', 'max_edit_distance']].copy()
feat_df['max_edit_distance'] = pd.to_numeric(feat_df['max_edit_distance'], errors='coerce')

print('Mean string features per category:')
print(
    feat_df.groupby('category')[['n_unique_raw', 'n_unique_normalized', 'max_edit_distance']]
    .mean().round(2).reindex(CAT_ORDER).to_string()
)

strip_chart = alt.Chart(feat_df).mark_tick(
    thickness=1.5, opacity=0.35,
).encode(
    x=alt.X('n_unique_raw:Q', title='N unique raw translations'),
    y=alt.Y('category:N', sort=CAT_ORDER, title=None),
    color=alt.Color(
        'category:N',
        scale=alt.Scale(domain=list(CAT_COLORS.keys()), range=list(CAT_COLORS.values())),
        legend=None,
    ),
    tooltip=['category:N', 'n_unique_raw:Q', 'max_edit_distance:Q'],
).properties(width=460, height=200, title='Unique raw translation count per category')

box_chart = alt.Chart(
    feat_df[
        feat_df['max_edit_distance'].notna() &
        (feat_df['max_edit_distance'] > 0)
    ]
).mark_boxplot(extent=1.5).encode(
    x=alt.X('max_edit_distance:Q', title='Max pairwise Levenshtein distance'),
    y=alt.Y('category:N', sort=CAT_ORDER, title=None),
    color=alt.Color(
        'category:N',
        scale=alt.Scale(domain=list(CAT_COLORS.keys()), range=list(CAT_COLORS.values())),
        legend=None,
    ),
).properties(
    width=460, height=200,
    title='Edit distance per category (zero-distance rows excluded)',
)

alt.vconcat(strip_chart, box_chart).resolve_scale(color='shared')

## 5.6 Category × Language Family

Does the disagreement pattern vary systematically by language family? Rows are sorted by `TRANSMOGRIFICATION` rate (highest first) to show where the residual unanchored-divergence bucket is most common.

This view should be read cautiously. Family-level differences here may reflect uneven Wikipedia coverage, differing rates of visible borrowing, or model behavior across scripts and training exposure, not a single linguistic property of the family itself.



In [ ]:
# Short display labels for the chart. Most reconciled family names (Atlantic-Congo,
# Mande, Algic, Turkic, ...) are already short, so a generic " languages" suffix-strip
# handles the long ISO 639-5 forms that survived reconciliation (Indo-European
# languages, Sino-Tibetan languages, Tai-Kadai languages). A small override dict
# handles the few cases that want explicit abbreviation or renaming.
FAMILY_ABBREVIATIONS = {
    'North American Indian':   'N. American Indian',
    'South American Indian':   'S. American Indian',
    'Central American Indian': 'C. American Indian',
    'Creoles and pidgins':     'Creoles/Pidgins',
    'Language isolate':        'Isolates',
}

def short_family_name(name):
    if not isinstance(name, str):
        return name
    stripped = name.removesuffix(' languages').removesuffix(' language')
    return FAMILY_ABBREVIATIONS.get(stripped, stripped)

# Restrict to families with at least 5 languages for readability
family_sizes = result_df['language_family'].value_counts()
major_families = family_sizes[family_sizes >= 5].index
plot_df = result_df[result_df['language_family'].isin(major_families)].copy()
plot_df['family_short'] = plot_df['language_family'].map(short_family_name)

family_cat = (
    plot_df.groupby(['family_short', 'category'])
    .size()
    .reset_index(name='n')
)
totals = family_cat.groupby('family_short')['n'].sum().rename('total')
family_cat = family_cat.join(totals, on='family_short')
family_cat['pct'] = (family_cat['n'] / family_cat['total'] * 100).round(1)

# Sort families by TRANSMOGRIFICATION rate
tmog_pct = (
    family_cat[family_cat['category'] == 'TRANSMOGRIFICATION']
    .set_index('family_short')['pct']
)
family_order = list(tmog_pct.sort_values(ascending=False).index)

print('TRANSMOGRIFICATION rate by family (top 10):')
for fam in family_order[:10]:
    n_total = int(totals.get(fam, 0))
    pct = tmog_pct.get(fam, 0)
    print(f'  {fam:<22} {pct:.1f}%  (n={n_total})')

alt.Chart(family_cat).mark_rect().encode(
    x=alt.X('category:N', sort=CAT_ORDER, title=None,
            axis=alt.Axis(labelAngle=-30, labelLimit=220)),
    y=alt.Y('family_short:N', sort=family_order, title=None),
    color=alt.Color('pct:Q', scale=alt.Scale(scheme='blues'), title='% of family'),
    tooltip=[
        'family_short:N', 'category:N',
        alt.Tooltip('n:Q', title='Languages'),
        alt.Tooltip('pct:Q', format='.1f', title='% of family'),
    ],
).properties(
    width=520, height=380,
    title=alt.TitleParams(
        'Disagreement category × language family',
        subtitle='% of each family in each category (families sorted by TRANSMOGRIFICATION rate)',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)

## 5.7 STRUCTURAL_ABSENCE — Borrowing Detail

This category is triggered whenever at least one service outputs an unadapted source-term token. That makes it analytically useful, but also broad. Some cases look like strong evidence of concept borrowing across many services; others are driven by a single service falling back to the source term while the rest attempt localization.

The summaries below make that internal variation explicit. Source tokens are derived automatically from the translation term, so for "Digital Humanities" we check for `digital`, `humanities`, `dh`, and the full expression. This remains a crude heuristic, but it helps distinguish widespread visible borrowing from isolated fallback behavior.



In [ ]:
absence_df = result_df[result_df['category'] == 'STRUCTURAL_ABSENCE'].copy()
print(f'STRUCTURAL_ABSENCE: {len(absence_df)} languages')

source_tokens = derive_source_tokens('Digital Humanities')

# Parse semicolon-separated loan_words_found
token_rows = []
borrow_detail_rows = []
for _, row in absence_df.iterrows():
    raw = str(row.get('loan_words_found', '') or '')
    matched_tokens = []
    if raw.strip() and raw not in ('nan', 'None'):
        for tok in raw.split(';'):
            tok = tok.strip()
            if tok:
                matched_tokens.append(tok)
                token_rows.append({
                    'language_code':   row['language_code'],
                    'language_name':   row['language_name'],
                    'language_family': row['language_family'],
                    'token': tok,
                })

    parsed = row.get('service_translations', {})
    if isinstance(parsed, str):
        try:
            parsed = ast.literal_eval(parsed)
        except Exception:
            parsed = {}

    borrowing_services = []
    for svc, text in parsed.items():
        hits = find_loan_words(text, source_tokens)
        if hits:
            borrowing_services.append(svc)

    n_borrowing_services = len(borrowing_services)
    if n_borrowing_services <= 1:
        borrow_band = '1 service'
    elif n_borrowing_services <= 3:
        borrow_band = '2–3 services'
    else:
        borrow_band = '4+ services'

    token_profile = (
        'both source words'
        if {'digital', 'humanities'}.issubset(set(matched_tokens))
        else 'digital only' if 'digital' in matched_tokens
        else 'humanities only' if 'humanities' in matched_tokens
        else 'other / acronym'
    )

    borrow_detail_rows.append({
        'language_code': row['language_code'],
        'language_name': row['language_name'],
        'language_family': row['language_family'],
        'n_borrowing_services': n_borrowing_services,
        'borrow_band': borrow_band,
        'token_profile': token_profile,
        'borrowing_services': '; '.join(borrowing_services),
    })

borrow_detail_df = pd.DataFrame(borrow_detail_rows)

if token_rows:
    token_df   = pd.DataFrame(token_rows)
    tok_counts = token_df['token'].value_counts().reset_index()
    tok_counts.columns = ['token', 'n']

    band_counts = (
        borrow_detail_df['borrow_band'].value_counts()
        .rename_axis('borrow_band').reset_index(name='n')
    )
    band_order = ['1 service', '2–3 services', '4+ services']
    band_counts['pct'] = (band_counts['n'] / len(absence_df) * 100).round(1)

    profile_counts = (
        borrow_detail_df['token_profile'].value_counts()
        .rename_axis('token_profile').reset_index(name='n')
    )

    print(f'Languages with detected loan words: {token_df["language_code"].nunique()}')
    print()
    print('How many services are visibly borrowing?')
    print(band_counts.set_index('borrow_band').reindex(band_order).fillna(0).to_string())
    print()
    print('Matched token profile:')
    print(profile_counts.to_string(index=False))
    print()
    print('Sample single-service borrowing cases:')
    single_service = borrow_detail_df[borrow_detail_df['borrow_band'] == '1 service']
    display(single_service[['language_code', 'language_name', 'language_family', 'borrowing_services', 'token_profile']].head(15))

    token_chart = alt.Chart(tok_counts).mark_bar(color='#9e9ac8').encode(
        x=alt.X('n:Q', title='Languages borrowing this token'),
        y=alt.Y('token:N', sort='-x', title=None),
        tooltip=['token:N', 'n:Q'],
    ).properties(
        width=320, height=160,
        title='Matched source tokens in STRUCTURAL_ABSENCE',
    )

    band_chart = alt.Chart(band_counts).mark_bar(color='#756bb1').encode(
        x=alt.X('n:Q', title='Languages'),
        y=alt.Y('borrow_band:N', sort=band_order, title=None),
        tooltip=['borrow_band:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='%')],
    ).properties(
        width=320, height=160,
        title='Borrowing-service count within STRUCTURAL_ABSENCE',
    )

    alt.hconcat(token_chart, band_chart)
else:
    print('No loan-word tokens detected in this run.')
    print('All STRUCTURAL_ABSENCE cases were classified by rule_no_translations.')
    print('Check SOURCE_TOKENS in explore_disagreements.py if unexpected.')



## 5.8 TRANSMOGRIFICATION — Unanchored Divergence

This is the residual category: multiple distinct outputs, no Wikipedia anchor, no detectable source-term borrowing. It marks cases that remain hard to anchor with the available evidence, not a single homogeneous error type.

Some entries in this bin are strongly divergent across many services. Others show partial clustering with a few outliers, or plausible local terms that simply lack a Wikipedia reference point. The summaries below therefore treat `TRANSMOGRIFICATION` as a heterogeneous space of unanchored divergence rather than as a definitive verdict on translation quality.



In [ ]:
tmog_df = result_df[result_df['category'] == 'TRANSMOGRIFICATION'].copy()
tmog_df['max_edit_distance'] = pd.to_numeric(tmog_df['max_edit_distance'], errors='coerce')
tmog_df['n_unique_raw'] = pd.to_numeric(tmog_df['n_unique_raw'], errors='coerce')

tmog_df['divergence_band'] = pd.cut(
    tmog_df['n_unique_raw'],
    bins=[0, 4, 6, 20],
    labels=['3–4 unique outputs', '5–6 unique outputs', '7+ unique outputs'],
)

band_counts = (
    tmog_df['divergence_band'].value_counts()
    .rename_axis('divergence_band').reset_index(name='n')
)
band_order = ['3–4 unique outputs', '5–6 unique outputs', '7+ unique outputs']
band_counts['pct'] = (band_counts['n'] / len(tmog_df) * 100).round(1)

print(f'TRANSMOGRIFICATION: {len(tmog_df)} languages ({len(tmog_df)/len(result_df)*100:.1f}%)')
print()
print('n_unique_raw distribution:')
print(tmog_df['n_unique_raw'].describe().round(2).to_string())
print()
print(f'Wikipedia present in any TRANSMOGRIFICATION case: {tmog_df["has_wikipedia"].any()}')
print()
print('Internal split within TRANSMOGRIFICATION:')
print(band_counts.set_index('divergence_band').reindex(band_order).fillna(0).to_string())
print()
print('This bin is not uniform: lower-divergence cases may still reflect partial convergence, while the 7+ output cases are the clearest examples of highly unstable naming.')
print()
print('Top 20 most-divergent cases (by n_unique_raw):')
sample_cols = ['language_code', 'language_name', 'language_family', 'n_unique_raw', 'max_edit_distance']
print(tmog_df.nlargest(20, 'n_unique_raw')[sample_cols].to_string(index=False))

hist = alt.Chart(tmog_df).mark_bar(color='#de2d26', opacity=0.8).encode(
    x=alt.X('n_unique_raw:Q', bin=alt.Bin(maxbins=10),
            title='N unique raw translations'),
    y=alt.Y('count()', title='Languages'),
    tooltip=[alt.Tooltip('n_unique_raw:Q', bin=True), 'count()'],
).properties(
    width=320, height=220,
    title='Output diversity within TRANSMOGRIFICATION',
)

band_chart = alt.Chart(band_counts).mark_bar(color='#fb6a4a').encode(
    x=alt.X('n:Q', title='Languages'),
    y=alt.Y('divergence_band:N', sort=band_order, title=None),
    tooltip=['divergence_band:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='%')],
).properties(
    width=320, height=220,
    title='Coarse divergence bands within TRANSMOGRIFICATION',
)

sim_chart = alt.Chart(
    tmog_df.dropna(subset=['mean_rationale_similarity'])
).mark_point(opacity=0.4, size=30, color='#de2d26').encode(
    x=alt.X('n_unique_raw:Q', title='N unique raw translations'),
    y=alt.Y('mean_rationale_similarity:Q', title='Mean rationale similarity'),
    tooltip=['language_name:N', 'n_unique_raw:Q',
             alt.Tooltip('mean_rationale_similarity:Q', format='.3f')],
).properties(
    width=320, height=220,
    title=alt.TitleParams(
        'Translation divergence vs. rationale similarity',
        subtitle='TRANSMOGRIFICATION preview of the notebook 07 two-axis analysis',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)

alt.hconcat(hist, band_chart, sim_chart)


## 5.9 Rebuild Downstream Data

The updated `disagreement_analysis.csv` should be propagated to the downstream data layer before running notebooks 06 or 07.

This step runs `build_disagreement_explorer_data.py`, which merges disagreement analysis, per-language confidence stats, and parsed service translations into `disagreement_explorer_data.csv`. That CSV is consumed by notebook 07 (rationale classification); it carries the heuristic bucket labels forward as one analytical field rather than as a final adjudication of translation quality.



In [ ]:
import subprocess

repo_root = str(Path('..').resolve())
proc = subprocess.run(
    ['python3', 'scripts/exploration/build_disagreement_explorer_data.py'],
    capture_output=True, text=True, cwd=repo_root,
)
print(proc.stdout)
if proc.returncode != 0:
    print('STDERR:', proc.stderr[:800])